# 🏆 Dual GLM Residual Booster Blend: Ensembling the Top Two Champions (Kaggle PGS S6E11)

Welcome to the **Dual GLM Residual Booster Blend** notebook for **Kaggle Playground Series Season 6 Episode 11 (Predicting Credit Card Fraud)**.

### 🎯 Motivation & Architecture
In our previous experiments, the two top-performing model architectures on the Kaggle Leaderboard were:
1. **LightGBM GLM Residual Booster**: **`0.94782`** Public LB (OOF PR-AUC: `0.51306`)
2. **XGBoost GLM Residual Booster**: **`0.94780`** Public LB (OOF PR-AUC: `0.51504`)

Both models leverage convex logistic regression log-odds margins ($z_i = \mathbf{w}^T \mathbf{x}_i + b$), but diverge in their tree building philosophy:
- **LightGBM**: Fast leaf-wise (best-first) histogram splitting.
- **XGBoost**: Robust depth-wise (level-wise) GPU histogram splitting.

By combining these two top models, we neutralize individual tree variance and create an ultra-calibrated ensemble aimed directly at **breaking past `0.9480+`**.

In [ ]:
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import rankdata, spearmanr, pearsonr
from scipy.optimize import minimize
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries imported successfully!")

In [ ]:
class Config:
    # Path resolution
    if os.path.exists('/kaggle/input/playground-series-s6e11'):
        DATA_DIR = Path('/kaggle/input/playground-series-s6e11')
        OUTPUT_DIR = Path('.')
    elif os.path.exists('/kaggle/input/competitions/playground-series-s6e11'):
        DATA_DIR = Path('/kaggle/input/competitions/playground-series-s6e11')
        OUTPUT_DIR = Path('.')
    else:
        DATA_DIR = Path('../../dataset')
        if not (DATA_DIR / 'train.csv').exists():
            DATA_DIR = Path('pgs-s6e11/dataset')
        OUTPUT_DIR = Path('../../submission')
        if not OUTPUT_DIR.exists():
            OUTPUT_DIR = Path('pgs-s6e11/submission')

Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data Directory:   {Config.DATA_DIR}")
print(f"Output Directory: {Config.OUTPUT_DIR}")

def locate_file(filename, candidate_roots):
    for root in candidate_roots:
        if root.exists():
            for p in root.rglob(filename):
                return p
    return None

search_roots = [
    Path('/kaggle/input'),
    Path('..'),
    Path('pgs-s6e11/code'),
    Path('.')
]

In [ ]:
print("Loading ground truth labels and sample submission...")
train_df = pd.read_csv(Config.DATA_DIR / 'train.csv', usecols=['id', 'is_fraud'])
test_df = pd.read_csv(Config.DATA_DIR / 'test.csv', usecols=['id'])
sample_sub = pd.read_csv(Config.DATA_DIR / 'sample_submission.csv')

y_true = train_df['is_fraud'].astype(int).values

# Locate OOF prediction arrays
lgb_oof_path = locate_file('oof_glm_boost.npy', search_roots)
xgb_oof_path = locate_file('oof_xgb_glm_boost.npy', search_roots)

print(f"LGBM GLM OOF path: {lgb_oof_path}")
print(f"XGB  GLM OOF path: {xgb_oof_path}")

assert lgb_oof_path is not None, "Could not find oof_glm_boost.npy!"
assert xgb_oof_path is not None, "Could not find oof_xgb_glm_boost.npy!"

oof_lgb = np.load(lgb_oof_path)
oof_xgb = np.load(xgb_oof_path)

# Locate test submission CSVs
lgb_sub_path = locate_file('glm_boost_submission.csv', search_roots)
xgb_sub_path = locate_file('xgb_glm_boost_submission.csv', search_roots)

print(f"LGBM GLM Sub path: {lgb_sub_path}")
print(f"XGB  GLM Sub path: {xgb_sub_path}")

assert lgb_sub_path is not None, "Could not find glm_boost_submission.csv!"
assert xgb_sub_path is not None, "Could not find xgb_glm_boost_submission.csv!"

sub_lgb = pd.read_csv(lgb_sub_path)
sub_xgb = pd.read_csv(xgb_sub_path)

test_lgb = sub_lgb['is_fraud'].values
test_xgb = sub_xgb['is_fraud'].values

print(f"\nLoaded {len(y_true):,} OOF samples and {len(test_lgb):,} Test predictions successfully!")

In [ ]:
auc_lgb = roc_auc_score(y_true, oof_lgb)
pr_lgb = average_precision_score(y_true, oof_lgb)

auc_xgb = roc_auc_score(y_true, oof_xgb)
pr_xgb = average_precision_score(y_true, oof_xgb)

print("=" * 65)
print("INDIVIDUAL MODEL VERIFICATION ON 5-FOLD OOF:")
print(f"  LightGBM GLM Residual Booster: ROC-AUC = {auc_lgb:.5f} | PR-AUC = {pr_lgb:.5f} (LB: 0.94782)")
print(f"  XGBoost  GLM Residual Booster: ROC-AUC = {auc_xgb:.5f} | PR-AUC = {pr_xgb:.5f} (LB: 0.94780)")
print("=" * 65)

# Correlation analysis
p_corr, _ = pearsonr(oof_lgb, oof_xgb)
s_corr, _ = spearmanr(oof_lgb, oof_xgb)

print(f"Prediction Correlation between LGB and XGB Residual Boosters:")
print(f"  Pearson Correlation:  {p_corr:.5f}")
print(f"  Spearman Rank Corr:   {s_corr:.5f}")
print(f"  (Diverse tree implementations provide room for ensembling synergy!)")

In [ ]:
# Normalize to percentile ranks
n_train = len(y_true)
n_test = len(test_lgb)

rank_oof_lgb = rankdata(oof_lgb) / n_train
rank_oof_xgb = rankdata(oof_xgb) / n_train

rank_test_lgb = rankdata(test_lgb) / n_test
rank_test_xgb = rankdata(test_xgb) / n_test

# Weight Sweep for Rank Average: w * LGB + (1-w) * XGB
weights = np.linspace(0.0, 1.0, 101)
sweep_rank_auc = []
sweep_prob_auc = []

for w in weights:
    blend_rank = w * rank_oof_lgb + (1 - w) * rank_oof_xgb
    blend_prob = w * oof_lgb + (1 - w) * oof_xgb
    
    sweep_rank_auc.append(roc_auc_score(y_true, blend_rank))
    sweep_prob_auc.append(roc_auc_score(y_true, blend_prob))

best_rank_idx = np.argmax(sweep_rank_auc)
best_prob_idx = np.argmax(sweep_prob_auc)

w_rank_best = weights[best_rank_idx]
auc_rank_best = sweep_rank_auc[best_rank_idx]

w_prob_best = weights[best_prob_idx]
auc_prob_best = sweep_prob_auc[best_prob_idx]

print("-" * 65)
print("BLENDING RESULTS & WEIGHT OPTIMIZATION:")
print(f"1. 50/50 Equal Rank Blend:     ROC-AUC = {roc_auc_score(y_true, 0.5*rank_oof_lgb + 0.5*rank_oof_xgb):.5f}")
print(f"2. Optimal Rank Blend:          ROC-AUC = {auc_rank_best:.5f} (Weight: {w_rank_best*100:.1f}% LGB, {(1-w_rank_best)*100:.1f}% XGB)")
print(f"3. 50/50 Equal Prob Blend:     ROC-AUC = {roc_auc_score(y_true, 0.5*oof_lgb + 0.5*oof_xgb):.5f}")
print(f"4. Optimal Prob Blend:          ROC-AUC = {auc_prob_best:.5f} (Weight: {w_prob_best*100:.1f}% LGB, {(1-w_prob_best)*100:.1f}% XGB)")
print("-" * 65)

# Calculate PR-AUC for optimal rank blend
opt_rank_oof = w_rank_best * rank_oof_lgb + (1 - w_rank_best) * rank_oof_xgb
opt_rank_pr = average_precision_score(y_true, opt_rank_oof)

print(f"Optimal Blend PR-AUC:           {opt_rank_pr:.5f}")
print(f"Gain over top standalone:       {auc_rank_best - max(auc_lgb, auc_xgb):+.5f} ROC-AUC")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Weight Sweep Curve
axes[0].plot(weights, sweep_rank_auc, color='#1565c0', lw=2.5, label='Rank Average Blend')
axes[0].plot(weights, sweep_prob_auc, color='#e65100', lw=2, linestyle='--', label='Probability Blend')
axes[0].axvline(x=w_rank_best, color='green', linestyle=':', label=f'Optimal Rank Weight ({w_rank_best:.2f})')
axes[0].axhline(y=max(auc_lgb, auc_xgb), color='gray', linestyle='--', label=f'Best Single Model ({max(auc_lgb, auc_xgb):.5f})')
axes[0].set_xlabel('Weight on LightGBM GLM Booster (w)')
axes[0].set_ylabel('5-Fold Out-of-Fold ROC-AUC')
axes[0].set_title('Blend Optimization Curve (OOF ROC-AUC vs Weight)')
axes[0].legend(loc='lower center')

# Plot 2: ROC Curve Comparison
fpr_lgb, tpr_lgb, _ = roc_curve(y_true, oof_lgb)
fpr_xgb, tpr_xgb, _ = roc_curve(y_true, oof_xgb)
fpr_blend, tpr_blend, _ = roc_curve(y_true, opt_rank_oof)

axes[1].plot(fpr_lgb, tpr_lgb, color='#0277bd', lw=1.5, alpha=0.7, label=f'LightGBM GLM (AUC = {auc_lgb:.5f})')
axes[1].plot(fpr_xgb, tpr_xgb, color='#f57c00', lw=1.5, alpha=0.7, label=f'XGBoost GLM (AUC = {auc_xgb:.5f})')
axes[1].plot(fpr_blend, tpr_blend, color='#2e7d32', lw=2.5, label=f'Dual GLM Blend (AUC = {auc_rank_best:.5f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle=':', alpha=0.5)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Dual GLM Residual Booster ROC Comparison')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.savefig('dual_glm_blend_diagnostics.png', dpi=150)
plt.show()

In [ ]:
# Compute final test predictions using optimal rank blend
final_test_preds = w_rank_best * rank_test_lgb + (1 - w_rank_best) * rank_test_xgb

submission = pd.DataFrame({
    'id': test_df['id'],
    'is_fraud': final_test_preds
})

assert submission.shape == sample_sub.shape, f"Shape mismatch: {submission.shape} vs {sample_sub.shape}"
assert list(submission.columns) == ['id', 'is_fraud'], f"Column mismatch: {submission.columns}"
assert not submission.isna().any().any(), "Submission contains null/NaN values!"
assert (submission['is_fraud'] >= 0.0).all() and (submission['is_fraud'] <= 1.0).all(), "Predictions out of [0, 1] range!"
assert (submission['id'] == sample_sub['id']).all(), "ID alignment mismatch!"

sub_path = Config.OUTPUT_DIR / 'dual_glm_blend_submission.csv'
submission.to_csv(sub_path, index=False)
submission.to_csv(Config.OUTPUT_DIR / 'submission.csv', index=False)

print("=" * 60)
print(f"FINAL SUBMISSION GENERATED AND VERIFIED AT: {sub_path}")
print(f"File size: {os.path.getsize(sub_path) / (1024*1024):.2f} MB")
print("=" * 60)
print("\nFirst 10 submission rows:")
print(submission.head(10))